# PPC v12.2 — Stable Gap-Focused Point Cloud Prediction

## Design philosophy: eliminate ALL NaN sources, then add complexity

| v12/v12.1 Issue | v12.2 Fix |
|---|---|
| AMP fp16 overflow → NaN gradients from batch 1 | **AMP disabled** — fp32 everywhere, no GradScaler |
| `normal_consistency_loss` eigh/cross-product → NaN backward | **Removed** → `smoothness_loss` (k-NN Laplacian, trivially stable) |
| `repulsion` exp(-da) → gradient explosion | **Removed** → `spacing_loss` (ReLU penalty, bounded gradient) |
| `hausdorff95_loss` large values → gradient issues | **Removed entirely** (use as eval metric only) |
| `compactness_loss` padded zeros → degenerate distances | **Removed** (redundant with Chamfer + gap) |
| `zgap_loss` .long() cast → ZERO gradient (dead loss!) | **Fixed** — differentiable soft Gaussian lookup |
| 16 losses from ep 1 → chaotic gradient landscape | **3-phase ramp-in**: 6→11→14 losses |
| `proj_loss` unbounded pixel distances | **Normalized** by IMG_SIZE² |
| KDE eps too small (1e-8) → divide-by-zero | **eps=1e-2** everywhere |


In [7]:
"""
PPC v12.2 — Stable Gap-Focused Training (AMP OFF, phased losses)
================================================================
Architecture : v8 proven MLP refinement (Encoder2D → FeatureLift →
               BiplanarFusion → CoarseUNet3D → QueryInit → 3× RefinementStage)
"""
import os, sys, json, time, random, warnings, csv, math
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import vtk
from tqdm import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    torch.cuda.set_per_process_memory_fraction(min(50.0 / total_gb, 0.95), 0)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

# ── PATHS ─────────────────────────────────────────────────────────────────────
DATA_ROOT   = Path("/apps/app/see_all_ai/dataset/Lumbar_Filtered_1037")
SPLIT_FILE  = DATA_ROOT / "dataset_split.json"
PROJECT_DIR = Path("/apps/app/pandu/ppc_network_v12_2")
CKPT_DIR    = PROJECT_DIR / "checkpoints"
LOG_DIR     = PROJECT_DIR / "logs"
RESULTS_DIR = PROJECT_DIR / "results"
for d in [PROJECT_DIR, CKPT_DIR, LOG_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── MODEL CONFIG ──────────────────────────────────────────────────────────────
IMG_SIZE            = 512
COARSE_VOL_SIZE     = 32
AUX_VOL_SIZE        = 48
GAP_VOL_SIZE        = 160
N_POINTS            = 5120
ENC_CHANNELS        = 192
VOL_CHANNELS        = 128
DEC_CHANNELS        = 128
QUERY_DIM           = 256
N_REFINE_STAGES     = 3
PRETRAINED_IMAGENET = True

# ── TRAINING CONFIG ───────────────────────────────────────────────────────────
BATCH_SIZE          = 2
NUM_WORKERS         = 4
EPOCHS              = 300
LR                  = 1e-4
WEIGHT_DECAY        = 1e-4
WARMUP_EPOCHS       = 10
USE_AMP             = False   # ← DISABLED: eliminates ALL fp16 overflow issues
GRAD_CLIP           = 1.0
EARLY_STOP_PATIENCE = 60
FREEZE_ENC_EPOCHS   = 5

# ── LOSS WEIGHTS ──────────────────────────────────────────────────────────────
# Phase 1 (ep 1+): Core losses only
W_CHAMFER   = 1.0
W_GAP       = 5.0
W_AXIAL     = 0.8
W_ZGAP      = 2.0
W_AUX_OCC   = 0.05
W_OFFSET    = 0.0005

# Phase 2 (ep 5+): + structure losses
W_SW        = 0.3
W_PROJ      = 0.02
W_SMOOTH    = 0.10    # replaces W_NORMAL
W_SPACING   = 0.02    # replaces W_REPULSION
W_BOUNDARY  = 0.20

# Phase 3 (ep 15+): + gap-specific losses
W_IVG       = 1.0
W_VSEP      = 0.5
W_GWIDTH    = 1.0

PHASE2_EPOCH = 5
PHASE3_EPOCH = 15

MAX_EVAL     = 103
CKPT_PATH    = CKPT_DIR / "latest_checkpoint.pth"
BEST_CKPT    = CKPT_DIR / "best_checkpoint.pth"
TRAINING_LOG = LOG_DIR  / "training_log.json"

print("=" * 65)
print("PPC v12.2 — Stable, Gap-Focused, AMP OFF")
print("=" * 65)
for k, v in dict(
    AMP="DISABLED (fp32 everywhere)",
    STAGES=N_REFINE_STAGES, N_POINTS=N_POINTS, GAP_VOL=GAP_VOL_SIZE,
    LR=LR, GRAD_CLIP=GRAD_CLIP,
    WARMUP=WARMUP_EPOCHS, FREEZE=FREEZE_ENC_EPOCHS,
    PHASE1="ep 1+: chamfer+gap+axial+zgap+aux+offset",
    PHASE2=f"ep {PHASE2_EPOCH}+: +sw+proj+smooth+spacing+boundary",
    PHASE3=f"ep {PHASE3_EPOCH}+: +ivg+vsep+gwidth",
    PATIENCE=EARLY_STOP_PATIENCE, EPOCHS=EPOCHS,
).items():
    print(f"  {k:<18} = {v}")

Device : cuda
GPU    : NVIDIA A100-SXM4-80GB
PPC v12.2 — Stable, Gap-Focused, AMP OFF
  AMP                = DISABLED (fp32 everywhere)
  STAGES             = 3
  N_POINTS           = 5120
  GAP_VOL            = 160
  LR                 = 0.0001
  GRAD_CLIP          = 1.0
  WARMUP             = 10
  FREEZE             = 5
  PHASE1             = ep 1+: chamfer+gap+axial+zgap+aux+offset
  PHASE2             = ep 5+: +sw+proj+smooth+spacing+boundary
  PHASE3             = ep 15+: +ivg+vsep+gwidth
  PATIENCE           = 60
  EPOCHS             = 300


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# DATA UTILITIES
# ══════════════════════════════════════════════════════════════════════════════

def load_vtk_points(path):
    r = vtk.vtkPolyDataReader(); r.SetFileName(str(path)); r.Update()
    p = r.GetOutput()
    return np.array([p.GetPoint(i) for i in range(p.GetNumberOfPoints())], np.float32)

def save_vtk_points(points, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    vp = vtk.vtkPoints()
    for pt in points: vp.InsertNextPoint(float(pt[0]), float(pt[1]), float(pt[2]))
    vc = vtk.vtkCellArray()
    for i in range(len(points)): vc.InsertNextCell(1); vc.InsertCellPoint(i)
    pd = vtk.vtkPolyData(); pd.SetPoints(vp); pd.SetVerts(vc)
    w = vtk.vtkPolyDataWriter()
    w.SetFileName(str(path)); w.SetInputData(pd); w.SetFileTypeToASCII(); w.Write()

def load_drr_image(path, size=IMG_SIZE):
    img = Image.open(path).convert("L")
    if img.size != (size, size): img = img.resize((size, size), Image.BILINEAR)
    return np.array(img, np.float32) / 255.0

def load_projection_matrix(path):
    with open(path) as f: vals = [float(v) for v in f.read().split()]
    return np.array(vals, np.float32).reshape(3, 4)

def load_split(p=SPLIT_FILE):
    with open(p) as f: d = json.load(f)
    return d["train"], d["val"], d["test"]

def append_log(path, rec):
    log = {"records": []}
    if path.exists():
        with open(path) as f: log = json.load(f)
    log["records"].append(rec)
    tmp = path.with_suffix(".tmp")
    with open(tmp, "w") as f: json.dump(log, f, indent=2)
    tmp.replace(path)

def pts_to_local(pts, c, s): return (pts - c[None]) / (s[None] + 1e-6)
def local_to_world(pts, c, s): return pts * s[:, None, :] + c[:, None, :]

def compute_scale(gt):
    e = (gt.max(0) - gt.min(0)).astype(np.float32)
    s = e * 0.55 + np.array([20., 20., 30.], np.float32)
    return np.maximum(s, np.array([50., 50., 80.], np.float32))

def make_aux_occ(pl, vs=AUX_VOL_SIZE, d=1):
    p   = np.clip((np.asarray(pl, np.float32) + 1) * 0.5, 0, 0.999999)
    idx = np.clip(np.floor(p * vs).astype(np.int32), 0, vs - 1)
    o   = np.zeros((vs,) * 3, np.float32)
    for dx in range(-d, d + 1):
        for dy in range(-d, d + 1):
            for dz in range(-d, d + 1):
                o[np.clip(idx[:,0]+dx,0,vs-1),
                  np.clip(idx[:,1]+dy,0,vs-1),
                  np.clip(idx[:,2]+dz,0,vs-1)] = 1.0
    return o

def make_gap_occ(pl, vs=GAP_VOL_SIZE):
    p   = np.clip((np.asarray(pl, np.float32) + 1) * 0.5, 0, 0.999999)
    idx = np.clip(np.floor(p * vs).astype(np.int32), 0, vs - 1)
    o   = np.zeros((vs,) * 3, np.float32)
    o[idx[:,0], idx[:,1], idx[:,2]] = 1.0
    return o

def flip_P(P, s=IMG_SIZE):
    return np.array([[-1,0,s-1],[0,1,0],[0,0,1]], np.float32) @ P


class LumbarDataset(Dataset):
    def __init__(self, ids, root=DATA_ROOT, aug=False):
        self.ids = ids; self.root = Path(root); self.aug = aug
        self.norm = transforms.Normalize(mean=[0.485], std=[0.229])

    def __len__(self): return len(self.ids)

    def __getitem__(self, i):
        pid = self.ids[i]; d = self.root / pid
        dap = load_drr_image(d / "AP_0"  / "drr_AP_0.png")
        dlp = load_drr_image(d / "LP_90" / "drr_LP_90.png")
        Pap = load_projection_matrix(d / "AP_0"  / "P_AP_0.txt")
        Plp = load_projection_matrix(d / "LP_90" / "P_LP_90.txt")
        gt  = load_vtk_points(d / "gt_ppc.vtk")
        c   = gt.mean(0).astype(np.float32); s = compute_scale(gt)
        gl  = np.clip(pts_to_local(gt, c, s), -1, 1)
        ao  = make_aux_occ(gl); go = make_gap_occ(gl)
        n   = len(gt)
        sel = np.random.choice(n, N_POINTS, replace=(n < N_POINTS))
        if self.aug:
            for x in [dap, dlp]:
                x[:] = np.clip(x * (1 + (np.random.rand()*2-1)*0.15), 0, 1)
            if np.random.rand() < 0.5:
                dap = dap[:, ::-1].copy(); dlp = dlp[:, ::-1].copy()
                Pap = flip_P(Pap); Plp = flip_P(Plp)
        return {
            "drr_ap":       self.norm(torch.from_numpy(dap).unsqueeze(0).float()),
            "drr_lp":       self.norm(torch.from_numpy(dlp).unsqueeze(0).float()),
            "P_ap":         torch.from_numpy(Pap).float(),
            "P_lp":         torch.from_numpy(Plp).float(),
            "gt_ppc_world": torch.from_numpy(gt[sel]).float(),
            "gt_ppc_local": torch.from_numpy(gl[sel]).float(),
            "aux_occ":      torch.from_numpy(ao).float(),
            "gap_occ":      torch.from_numpy(go).float(),
            "center":       torch.from_numpy(c).float(),
            "scale":        torch.from_numpy(s).float(),
            "patient_id":   pid,
        }

train_ids, val_ids, test_ids = load_split()
print(f"Split: train={len(train_ids)}  val={len(val_ids)}  test={len(test_ids)}")

Split: train=829  val=103  test=105


In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# MODEL — v8 proven architecture
# ══════════════════════════════════════════════════════════════════════════════

class Encoder2D(nn.Module):
    def __init__(self, in_ch=1, out_ch=ENC_CHANNELS, pretrained=PRETRAINED_IMAGENET):
        super().__init__()
        base = models.resnet18(
            weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        c1 = nn.Conv2d(in_ch, 64, 7, stride=2, padding=3, bias=False)
        if pretrained:
            with torch.no_grad():
                c1.weight[:] = base.conv1.weight.mean(1, keepdim=True)
        base.conv1 = c1
        self.stem   = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool)
        self.layer1 = base.layer1; self.layer2 = base.layer2; self.layer3 = base.layer3
        self.proj   = nn.Conv2d(256, out_ch, 1)
    def forward(self, x):
        return self.proj(self.layer3(self.layer2(self.layer1(self.stem(x)))))

class FeatureLift(nn.Module):
    def __init__(self, in_ch=ENC_CHANNELS, out_ch=VOL_CHANNELS, depth=COARSE_VOL_SIZE):
        super().__init__()
        self.depth_embed = nn.Parameter(torch.randn(1, in_ch, depth, 1, 1) * 0.02)
        self.refine = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1), nn.GroupNorm(8, out_ch), nn.GELU(),
            nn.Conv3d(out_ch, out_ch, 3, padding=1), nn.GroupNorm(8, out_ch), nn.GELU())
    def forward(self, f2d):
        B, C, H, W = f2d.shape
        vol = f2d.unsqueeze(2).expand(B, C, COARSE_VOL_SIZE, H, W).contiguous()
        return self.refine(vol + self.depth_embed)

class BiplanarFusion(nn.Module):
    def __init__(self, in_ch=VOL_CHANNELS, out_ch=VOL_CHANNELS):
        super().__init__()
        self.fuse = nn.Sequential(
            nn.Conv3d(in_ch*2, out_ch, 1), nn.GroupNorm(8, out_ch), nn.GELU(),
            nn.Conv3d(out_ch, out_ch, 3, padding=1), nn.GroupNorm(8, out_ch), nn.GELU())
    def forward(self, ap, lp):
        return self.fuse(torch.cat([
            ap.permute(0,1,4,2,3).contiguous(),
            lp.permute(0,1,2,4,3).contiguous()], 1))

class Block3D(nn.Module):
    def __init__(self, ic, oc):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(ic, oc, 3, padding=1), nn.GroupNorm(8, oc), nn.GELU(),
            nn.Conv3d(oc, oc, 3, padding=1), nn.GroupNorm(8, oc), nn.GELU())
    def forward(self, x): return self.block(x)

class CoarseUNet3D(nn.Module):
    def __init__(self, in_ch=VOL_CHANNELS, fc=DEC_CHANNELS):
        super().__init__()
        self.enc1 = Block3D(in_ch, 96); self.down1 = nn.Conv3d(96, 160, 2, stride=2)
        self.enc2 = Block3D(160, 160);  self.down2 = nn.Conv3d(160, 224, 2, stride=2)
        self.bottleneck = Block3D(224, 224)
        self.up2  = nn.ConvTranspose3d(224, 160, 2, stride=2); self.dec2 = Block3D(320, 160)
        self.up1  = nn.ConvTranspose3d(160,  96, 2, stride=2); self.dec1 = Block3D(192, fc)
        self.aux_head = nn.Sequential(
            nn.Conv3d(fc, fc//2, 3, padding=1), nn.GELU(), nn.Conv3d(fc//2, 1, 1))
    def forward(self, x):
        e1 = self.enc1(x); e2 = self.enc2(self.down1(e1))
        b  = self.bottleneck(self.down2(e2))
        d2 = self.dec2(torch.cat([self.up2(b), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        aux = F.interpolate(self.aux_head(d1), size=(AUX_VOL_SIZE,)*3,
                            mode="trilinear", align_corners=True)
        return d1, aux

def project_points(P, pts, s=IMG_SIZE):
    B, N, _ = pts.shape
    h   = torch.cat([pts, torch.ones(B, N, 1, device=pts.device, dtype=pts.dtype)], -1)
    uvw = torch.bmm(h, P.transpose(1, 2)); z = uvw[..., 2:3].clamp(min=1e-6)
    uv  = uvw[..., :2] / z
    return uv, (uv / (s-1)) * 2 - 1, torch.log(z)

def sample_2d(fm, uvn):
    return F.grid_sample(fm, uvn.view(fm.shape[0], -1, 1, 2), mode="bilinear",
                         align_corners=True, padding_mode="border").squeeze(-1).transpose(1, 2)

def sample_3d(vf, pl):
    g = torch.stack([pl[...,2], pl[...,1], pl[...,0]], -1).view(pl.shape[0], -1, 1, 1, 3)
    return F.grid_sample(vf, g, mode="bilinear", align_corners=True,
                         padding_mode="border").squeeze(-1).squeeze(-1).transpose(1, 2)

class QueryInit(nn.Module):
    def __init__(self, n=N_POINTS, fc=DEC_CHANNELS):
        super().__init__()
        grid = np.stack(np.meshgrid(np.linspace(-.8,.8,16), np.linspace(-.8,.8,16),
                                    np.linspace(-.9,.9,20), indexing="ij"), -1
                        ).reshape(-1, 3).astype(np.float32)
        self.register_buffer("base", torch.from_numpy(grid)); self.n = n
        self.head = nn.Sequential(nn.AdaptiveAvgPool3d(1), nn.Flatten(),
                                  nn.Linear(fc, 256), nn.GELU(), nn.Linear(256, n*3))
    def forward(self, vf, aux=None):
        B = vf.shape[0]; off = self.head(vf).view(B, self.n, 3)
        raw = self.base[None] + 0.20 * torch.tanh(off)
        if aux is not None:
            gs   = torch.stack([raw[...,2], raw[...,1], raw[...,0]], -1).view(B, self.n, 1, 1, 3)
            gate = torch.sigmoid(F.grid_sample(aux, gs, mode="bilinear", align_corners=True,
                                               padding_mode="border").view(B, self.n)).unsqueeze(-1)
            raw  = self.base[None] + gate * 0.25 * torch.tanh(off)
        return torch.clamp(raw, -1, 1)

class RefinementStage(nn.Module):
    def __init__(self, f2d=ENC_CHANNELS, f3d=DEC_CHANNELS, h=QUERY_DIM):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(f2d*2 + f3d + 3 + 2 + 2 + 1, h), nn.GELU(),
            nn.Linear(h, h), nn.GELU(), nn.Linear(h, 3))
    def forward(self, q, vf, fap, flp, Pap, Plp, c, s, aux=None):
        qw = local_to_world(q, c, s)
        _, ua, _ = project_points(Pap, qw)
        _, ul, _ = project_points(Plp, qw)
        B, N, _ = q.shape
        if aux is not None:
            gs = torch.stack([q[...,2], q[...,1], q[...,0]], -1).view(B, N, 1, 1, 3)
            ov = F.grid_sample(aux, gs, mode="bilinear", align_corners=True,
                               padding_mode="border").view(B, N, 1)
        else:
            ov = torch.zeros(B, N, 1, device=q.device)
        x     = torch.cat([sample_3d(vf, q), sample_2d(fap, ua), sample_2d(flp, ul), q, ua, ul, ov], -1)
        delta = 0.25 * torch.tanh(self.mlp(x))
        return q + delta, {"delta": delta}

class PPCNetV12(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder_ap = Encoder2D(); self.encoder_lp = Encoder2D()
        self.lift_ap    = FeatureLift(); self.lift_lp  = FeatureLift()
        self.fusion     = BiplanarFusion(); self.coarse3d = CoarseUNet3D()
        self.query_init = QueryInit()
        self.stages     = nn.ModuleList([RefinementStage() for _ in range(N_REFINE_STAGES)])
    def forward(self, dap, dlp, Pap, Plp, c, s):
        fap = self.encoder_ap(dap); flp = self.encoder_lp(dlp)
        vf, aux = self.coarse3d(self.fusion(self.lift_ap(fap), self.lift_lp(flp)))
        q = self.query_init(vf, aux); sa = []
        for stage in self.stages:
            q, a = stage(q, vf, fap, flp, Pap, Plp, c, s, aux); sa.append(a)
        return {"pred_local": torch.clamp(q, -1, 1),
                "pred_world": local_to_world(q, c, s),
                "aux_occ_logits": aux, "stage_aux": sa}

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
_m = PPCNetV12(); print(f"PPCNetV12 params: {count_params(_m)/1e6:.1f} M"); del _m

PPCNetV12 params: 21.8 M


In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# LOSS FUNCTIONS — v12.2: every function individually proven NaN-free
# ══════════════════════════════════════════════════════════════════════════════
#
# Design rules:
#   1. No eigendecomposition (eigh) — NaN gradients for degenerate inputs
#   2. No exp() on unbounded inputs — gradient explosion
#   3. No .long() / .int() casts in gradient path — kills gradient
#   4. Every loss output is bounded and has bounded gradient
#   5. eps >= 1e-2 for all divisions
#

def safe_loss(val, name='loss', cap=1e4):
    """NaN/Inf guard. Returns non-grad zero so loss sum still works."""
    if torch.isnan(val) or torch.isinf(val):
        print(f"  ⚠ NaN/Inf in {name}, returning 0")
        return torch.tensor(0.0, device=val.device)
    return torch.clamp(val, max=cap)


def pairwise_sq(x, y):
    """Batched pairwise squared Euclidean distances. (B,N,3),(B,M,3)→(B,N,M)"""
    x2 = (x ** 2).sum(-1, keepdim=True)
    y2 = (y ** 2).sum(-1).unsqueeze(1)
    return (x2 + y2 - 2.0 * torch.bmm(x, y.transpose(1, 2))).clamp_min(0)


# ── Phase 1: Core positional losses (ep 1+) ──────────────────────────────────

def chamfer_loss(pred, gt, wp=1.0, wg=1.5, tr=15.0):
    """Bidirectional Chamfer with truncation. Bounded by tr²."""
    d2 = pairwise_sq(pred, gt)
    fwd = torch.clamp(d2.min(2)[0], max=tr ** 2).mean()
    bwd = torch.clamp(d2.min(1)[0], max=tr ** 2).mean()
    return wp * fwd + wg * bwd


def gap_penalty(pl, gap_occ):
    """Penalize points in empty GT voxels. Output in [0, 1]."""
    B, N, _ = pl.shape
    occ = gap_occ.unsqueeze(1)  # (B, 1, D, H, W)
    g = torch.stack([pl[..., 2], pl[..., 1], pl[..., 0]], -1).view(B, N, 1, 1, 3)
    s = F.grid_sample(occ, g, mode="bilinear", align_corners=True,
                      padding_mode="border").view(B, N)
    return ((1.0 - s).clamp(min=0) ** 2).mean()


def axial_density(pl, gl, nb=80, sig=0.020):
    """Match Z-axis density distribution via KDE. Bounded by 2.0."""
    c  = torch.linspace(-1, 1, nb, device=pl.device)
    pk = torch.exp(-0.5 * ((pl[..., 2].unsqueeze(-1) - c) / sig) ** 2).sum(1)
    gk = torch.exp(-0.5 * ((gl[..., 2].unsqueeze(-1) - c) / sig) ** 2).sum(1)
    pd = pk / (pk.sum(1, keepdim=True) + 1e-2)
    gd = gk / (gk.sum(1, keepdim=True) + 1e-2)
    return (pd - gd).abs().sum(1).mean()


def zgap_loss(pl, gl, ns=120, sig=0.012):
    """
    Z-gap loss — FIXED in v12.2: fully differentiable soft lookup.
    v12 used .long() cast which killed ALL gradient (dead loss).
    Now uses Gaussian soft-attention over bins — proper gradient flow.
    """
    c = torch.linspace(-1, 1, ns, device=pl.device)

    # GT gap mask: 1 where GT density < 5% of peak
    gk = torch.exp(-0.5 * ((gl[..., 2].unsqueeze(-1) - c) / sig) ** 2).sum(1)  # (B, ns)
    gm = ((gk / (gk.max(1, keepdim=True)[0] + 1e-8)) < 0.05).float()          # (B, ns)

    # Soft lookup: pred point's Gaussian attention over bins
    pred_z = pl[..., 2]                                                         # (B, N)
    attn = torch.exp(-0.5 * ((pred_z.unsqueeze(-1) - c) / sig) ** 2)           # (B, N, ns)
    attn = attn / (attn.sum(-1, keepdim=True) + 1e-2)                          # normalize

    # Weighted sum: how much each pred point overlaps with gap zones
    gap_score = (attn * gm.unsqueeze(1)).sum(-1)                               # (B, N)
    return gap_score.mean()


def dice_loss(logits, tgt, eps=1e-6):
    p = torch.sigmoid(logits).reshape(logits.shape[0], -1)
    t = tgt.reshape(tgt.shape[0], -1)
    return 1.0 - ((2 * (p * t).sum(1) + eps) / (p.sum(1) + t.sum(1) + eps)).mean()


# ── Phase 2: Structure losses (ep PHASE2_EPOCH+) ─────────────────────────────

def sw_loss(pred, gt, np_=50):
    """Sliced Wasserstein distance. Bounded."""
    dirs = F.normalize(torch.randn(np_, 3, device=pred.device), -1)
    pp = torch.einsum("bnd,pd->bnp", pred, dirs)
    gp = torch.einsum("bnd,pd->bnp", gt, dirs)
    return ((pp.sort(1)[0] - gp.sort(1)[0]) ** 2).mean()


def proj_loss(pw, gw, Pap, Plp):
    """
    2D projection Chamfer — normalized by IMG_SIZE² to bound values in [0, 1].
    """
    def ch2d(a, b):
        d2 = pairwise_sq(a, b) / (IMG_SIZE ** 2 + 1e-6)
        return 0.5 * (torch.clamp(d2.min(2)[0], max=1.0).mean()
                    + torch.clamp(d2.min(1)[0], max=1.0).mean())
    pa, _, _ = project_points(Pap, pw); ga, _, _ = project_points(Pap, gw)
    pl, _, _ = project_points(Plp, pw); gl, _, _ = project_points(Plp, gw)
    return ch2d(pa, ga) + ch2d(pl, gl)


def smoothness_loss(pts, k=6):
    """
    k-NN Laplacian smoothness — replaces normal_consistency_loss.
    
    Why replaced: normal_consistency used torch.linalg.eigh (v12) or cross-product
    (v12.1) for local surface normals. Both produce NaN gradients for degenerate
    neighborhoods (coplanar, collinear, or overlapping points).
    
    This loss simply penalizes each point's deviation from its local k-NN mean.
    Gradient: 2*(pt - local_mean) / N — always finite, always bounded.
    """
    B, N, _ = pts.shape
    d2 = pairwise_sq(pts, pts)
    eye = torch.eye(N, device=pts.device, dtype=torch.bool).unsqueeze(0)
    d2 = d2.masked_fill(eye, float('inf'))
    knn_idx = d2.topk(k, dim=-1, largest=False).indices       # (B, N, k)
    nbrs = torch.gather(
        pts.unsqueeze(2).expand(B, N, N, 3), 2,
        knn_idx.unsqueeze(-1).expand(B, N, k, 3))              # (B, N, k, 3)
    local_mean = nbrs.mean(2)                                  # (B, N, 3)
    return ((pts - local_mean) ** 2).sum(-1).mean()


def spacing_loss(pts, k=4, target_sq=0.0004):
    """
    Minimum spacing penalty — replaces repulsion loss.
    
    Why replaced: repulsion used exp(-da/h²) which has gradient ~exp(-da)/h².
    For near-overlapping points (da≈0), h=0.02 → gradient ≈ 1/0.0004 = 2500.
    Even with da.clamp(min=0.01), the exp backward still produces huge values.
    
    This loss uses ReLU(target² - d²) — gradient is exactly -1 when active,
    exactly 0 when inactive. Always bounded, never NaN.
    target_sq = 0.02² = 0.0004 in local coords.
    """
    B, N, _ = pts.shape
    d2 = pairwise_sq(pts, pts)
    eye = torch.eye(N, device=pts.device, dtype=torch.bool).unsqueeze(0)
    d2 = d2.masked_fill(eye, float('inf'))
    knn_d2 = d2.topk(k, dim=-1, largest=False).values          # (B, N, k)
    return F.relu(target_sq - knn_d2).mean()


def boundary_chamfer_loss(pred_local, gap_occ_gt, vol_size=GAP_VOL_SIZE):
    """Penalise points NOT on GT voxel surface boundary. Output in [0, 1]."""
    B, N, _ = pred_local.shape
    occ      = gap_occ_gt.unsqueeze(1).float()
    dilated  = F.max_pool3d(occ, kernel_size=3, stride=1, padding=1)
    eroded   = -F.max_pool3d(-occ, kernel_size=3, stride=1, padding=1)
    boundary = (dilated - eroded).clamp(0, 1)
    g = torch.stack([pred_local[..., 2], pred_local[..., 1], pred_local[..., 0]], -1
                    ).view(B, N, 1, 1, 3)
    on_bnd = F.grid_sample(boundary, g, mode="bilinear", align_corners=True,
                           padding_mode="border").view(B, N)
    return (1.0 - on_bnd).mean()


# ── Phase 3: Intervertebral gap losses (ep PHASE3_EPOCH+) ────────────────────

def intervertebral_gap_loss(pred_local, gt_local, nb=120, sig=0.012):
    """Gap-zone repulsion via soft Gaussian KDE. Squared penalty."""
    B = pred_local.shape[0]
    centers = torch.linspace(-1, 1, nb, device=pred_local.device)

    gt_z = gt_local[..., 2]
    gt_kde = torch.exp(-0.5 * ((gt_z.unsqueeze(-1) - centers) / sig) ** 2).sum(1)
    gt_norm = gt_kde / (gt_kde.max(1, keepdim=True)[0] + 1e-8)
    gap_mask = (gt_norm < 0.05).float()

    pred_z = pred_local[..., 2]
    pred_kde = torch.exp(-0.5 * ((pred_z.unsqueeze(-1) - centers) / sig) ** 2)
    pred_kde_norm = pred_kde / (pred_kde.sum(-1, keepdim=True) + 1e-2)

    gap_occupancy = (pred_kde_norm * gap_mask.unsqueeze(1)).sum(-1)
    return (gap_occupancy ** 2).mean()


def vertebra_separation_loss(pred_local, gt_local, nb=120, sig=0.015):
    """Penalize pred density at GT valley locations (interior only)."""
    B = pred_local.shape[0]
    centers = torch.linspace(-1, 1, nb, device=pred_local.device)

    gt_z = gt_local[..., 2]
    gt_kde = torch.exp(-0.5 * ((gt_z.unsqueeze(-1) - centers) / sig) ** 2).sum(1)
    gt_norm = gt_kde / (gt_kde.max(1, keepdim=True)[0] + 1e-8)

    valley_mask = (gt_norm < 0.10).float()
    cum_left  = torch.cumsum(gt_norm, dim=1)
    cum_right = torch.cumsum(gt_norm.flip(1), dim=1).flip(1)
    interior_valley = valley_mask * (cum_left > 0.5).float() * (cum_right > 0.5).float()

    pred_z = pred_local[..., 2]
    pred_kde = torch.exp(-0.5 * ((pred_z.unsqueeze(-1) - centers) / sig) ** 2).sum(1)
    pred_norm = pred_kde / (pred_kde.max(1, keepdim=True)[0] + 1e-2)

    return (pred_norm * interior_valley).sum(1).mean()


def gap_width_preservation_loss(pred_local, gt_local, nb=120, sig=0.015):
    """Match Z-density profiles with 5x weight on gap regions."""
    B = pred_local.shape[0]
    centers = torch.linspace(-1, 1, nb, device=pred_local.device)

    gt_z = gt_local[..., 2]
    gt_kde = torch.exp(-0.5 * ((gt_z.unsqueeze(-1) - centers) / sig) ** 2).sum(1)
    gt_d = gt_kde / (gt_kde.sum(1, keepdim=True) + 1e-2)
    gt_norm = gt_kde / (gt_kde.max(1, keepdim=True)[0] + 1e-8)

    pred_z = pred_local[..., 2]
    pred_kde = torch.exp(-0.5 * ((pred_z.unsqueeze(-1) - centers) / sig) ** 2).sum(1)
    pred_d = pred_kde / (pred_kde.sum(1, keepdim=True) + 1e-2)

    gap_weight = 1.0 + 4.0 * (gt_norm < 0.15).float()
    return (gap_weight * (pred_d - gt_d).abs()).sum(1).mean()


# ── Combined loss with phasing ───────────────────────────────────────────────

def total_loss(out, batch, epoch=0):
    """
    v12.2 combined loss with 3-phase ramp-in.
    Phase 1 (ep 1+):  6 core losses — guaranteed stable
    Phase 2 (ep 5+):  +5 structure losses with 5-epoch ramp
    Phase 3 (ep 15+): +3 gap losses with 5-epoch ramp
    """
    pw, pl = out["pred_world"], out["pred_local"]
    gw, gl = batch["gt_ppc_world"], batch["gt_ppc_local"]

    # ── Phase 1: Core (always active) ──
    lc   = safe_loss(chamfer_loss(pw, gw), "chamfer")
    lg   = safe_loss(gap_penalty(pl, batch["gap_occ"]), "gap")
    la   = safe_loss(axial_density(pl, gl), "axial")
    lz   = safe_loss(zgap_loss(pl, gl), "zgap")
    laux = safe_loss(
        F.binary_cross_entropy_with_logits(
            out["aux_occ_logits"].squeeze(1), batch["aux_occ"])
        + dice_loss(out["aux_occ_logits"].squeeze(1), batch["aux_occ"]),
        "aux_occ")
    lo   = safe_loss(
        torch.stack([a["delta"].abs().mean() for a in out["stage_aux"]]).mean(),
        "offset")

    loss = (W_CHAMFER * lc + W_GAP * lg + W_AXIAL * la + W_ZGAP * lz
            + W_AUX_OCC * laux + W_OFFSET * lo)

    bd = {"chamfer": lc, "gap": lg, "axial": la, "zgap": lz,
          "aux_occ": laux, "offset": lo}

    # ── Phase 2: Structure (ramped in from PHASE2_EPOCH) ──
    if epoch >= PHASE2_EPOCH:
        ramp2 = min(1.0, (epoch - PHASE2_EPOCH + 1) / 5.0)
        ls   = safe_loss(sw_loss(pl, gl), "sw")
        lp   = safe_loss(proj_loss(pw, gw, batch["P_ap"], batch["P_lp"]), "proj")
        lsm  = safe_loss(smoothness_loss(pl), "smooth")
        lsp  = safe_loss(spacing_loss(pl), "spacing")
        lb   = safe_loss(boundary_chamfer_loss(pl, batch["gap_occ"]), "boundary")

        loss = loss + ramp2 * (W_SW * ls + W_PROJ * lp + W_SMOOTH * lsm
                              + W_SPACING * lsp + W_BOUNDARY * lb)
        bd.update({"sw": ls, "proj": lp, "smooth": lsm, "spacing": lsp, "boundary": lb})

    # ── Phase 3: Gap-specific (ramped in from PHASE3_EPOCH) ──
    if epoch >= PHASE3_EPOCH:
        ramp3 = min(1.0, (epoch - PHASE3_EPOCH + 1) / 5.0)
        livg  = safe_loss(intervertebral_gap_loss(pl, gl), "ivg")
        lvsep = safe_loss(vertebra_separation_loss(pl, gl), "vsep")
        lgw   = safe_loss(gap_width_preservation_loss(pl, gl), "gwidth")

        loss = loss + ramp3 * (W_IVG * livg + W_VSEP * lvsep + W_GWIDTH * lgw)
        bd.update({"ivg": livg, "vsep": lvsep, "gwidth": lgw})

    # Convert to float for logging
    bd_f = {"total": float(loss.detach())}
    for k, v in bd.items():
        bd_f[k] = float(v.detach()) if isinstance(v, torch.Tensor) else float(v)

    return loss, bd_f


print("All loss functions defined (v12.2 — stable, phased, 14 losses).")
print(f"  Phase 1 (ep 1+):  chamfer, gap, axial, zgap, aux_occ, offset")
print(f"  Phase 2 (ep {PHASE2_EPOCH}+):  + sw, proj, smooth, spacing, boundary")
print(f"  Phase 3 (ep {PHASE3_EPOCH}+): + ivg, vsep, gwidth")

All loss functions defined (v12.2 — stable, phased, 14 losses).
  Phase 1 (ep 1+):  chamfer, gap, axial, zgap, aux_occ, offset
  Phase 2 (ep 5+):  + sw, proj, smooth, spacing, boundary
  Phase 3 (ep 15+): + ivg, vsep, gwidth


In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# TRAINING LOOP — v12.2: fp32, phased losses, clean NaN handling
# ══════════════════════════════════════════════════════════════════════════════

def collate_fn(b):
    out = {}
    for k in b[0]:
        vals = [x[k] for x in b]
        out[k] = torch.stack(vals, 0) if isinstance(vals[0], torch.Tensor) else vals
    return out

def to_dev(b):
    return {k: (v.to(device, non_blocking=True) if isinstance(v, torch.Tensor) else v)
            for k, v in b.items()}

def chamfer_np(pred, gt):
    pred, gt = np.asarray(pred, np.float32), np.asarray(gt, np.float32)
    d = np.linalg.norm(pred[:, None] - gt[None], axis=-1)
    dp, dg = d.min(1), d.min(0)
    def fs(th): p, r = (dp<=th).mean(), (dg<=th).mean(); return 2*p*r/(p+r) if p+r>0 else 0.
    return {"chamfer_mm": float(0.5*(dp.mean()+dg.mean())),
            "fscore_1mm": float(fs(1)), "fscore_2mm": float(fs(2)),
            "fscore_5mm": float(fs(5)),
            "hausdorff_95": float(max(np.percentile(dp,95), np.percentile(dg,95)))}


# ── Build model ───────────────────────────────────────────────────────────────
model = PPCNetV12().to(device)
print(f"Params : {count_params(model)/1e6:.1f} M")
print("Training from SCRATCH — ImageNet encoder only.")

enc_params   = list(model.encoder_ap.parameters()) + list(model.encoder_lp.parameters())
enc_ids      = {id(p) for p in enc_params}
other_params = [p for p in model.parameters() if id(p) not in enc_ids]

optimizer = torch.optim.AdamW([
    {"params": enc_params,   "lr": LR * 0.1},
    {"params": other_params, "lr": LR},
], weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=50, T_mult=2, eta_min=1e-6)

# NO GradScaler — AMP is disabled
train_ds     = LumbarDataset(train_ids, aug=True)
val_ds       = LumbarDataset(val_ids,   aug=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          collate_fn=collate_fn, persistent_workers=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          collate_fn=collate_fn, persistent_workers=True)
print(f"Train  : {len(train_ds)} samples → {len(train_loader)} batches/ep")
print(f"Val    : {len(val_ds)} samples → {len(val_loader)} batches/ep")


def save_ckpt(path, ep, best, hist):
    tmp = path.with_suffix(".tmp")
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "epoch": ep, "best_chamfer": best, "history": hist,
                "config": {"version": "v12.2_stable"}}, tmp)
    tmp.replace(path)
    print(f"  Saved → {path.name}  (ep {ep+1})")


def load_ckpt(path):
    st = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(st["model"]); optimizer.load_state_dict(st["optimizer"])
    if st.get("scheduler"): scheduler.load_state_dict(st["scheduler"])
    ep = st["epoch"] + 1; bc = st.get("best_chamfer", float("inf")); h = st.get("history", [])
    print(f"  Resumed from ep {ep} | best={bc:.3f} mm")
    return ep, bc, h


start_epoch, best_chamfer, history, no_improve = 0, float("inf"), [], 0
collapse_count = 0

if CKPT_PATH.exists():
    print(f"Found checkpoint: {CKPT_PATH.name}")
    start_epoch, best_chamfer, history = load_ckpt(CKPT_PATH)
else:
    print("No checkpoint — starting fresh.")

print(f"{'='*65}")
print(f"v12.2 training from ep {start_epoch+1}/{EPOCHS}")
print(f"AMP=OFF | GRAD_CLIP={GRAD_CLIP} | fp32 everywhere")
print(f"Phase 1: core only | Phase 2: ep {PHASE2_EPOCH}+ | Phase 3: ep {PHASE3_EPOCH}+")
print(f"{'='*65}")


for epoch in range(start_epoch, EPOCHS):

    # ── Encoder freeze/unfreeze ──
    if epoch < FREEZE_ENC_EPOCHS:
        for p in enc_params: p.requires_grad_(False)
        optimizer.param_groups[0]["lr"] = 0.0
        if epoch == start_epoch: print(f"  [Encoder FROZEN for ep 1-{FREEZE_ENC_EPOCHS}]")
    elif epoch == FREEZE_ENC_EPOCHS:
        for p in enc_params: p.requires_grad_(True)
        optimizer.param_groups[0]["lr"] = LR * 0.1
        print(f"  [Encoder UNFROZEN at ep {epoch+1}]")

    # ── Phase announcement ──
    if epoch == PHASE2_EPOCH:
        print(f"  [PHASE 2] Adding: sw, proj, smooth, spacing, boundary")
    if epoch == PHASE3_EPOCH:
        print(f"  [PHASE 3] Adding: ivg, vsep, gwidth")

    # ── Training ──
    model.train(); t0 = time.time(); acc = {}; nb = 0; nan_count = 0
    pbar = tqdm(enumerate(train_loader, 1), total=len(train_loader),
                desc=f'Ep {epoch+1:3d}/{EPOCHS}', leave=True, ncols=120)

    for bi, batch in pbar:
        batch = to_dev(batch)

        # Warmup
        if epoch < WARMUP_EPOCHS:
            step = epoch * len(train_loader) + bi
            frac = max(0.05, step / (WARMUP_EPOCHS * len(train_loader)))
            optimizer.param_groups[1]["lr"] = LR * frac
            if epoch >= FREEZE_ENC_EPOCHS:
                optimizer.param_groups[0]["lr"] = LR * 0.1 * frac

        optimizer.zero_grad(set_to_none=True)

        # Forward — fp32, no autocast
        out  = model(batch["drr_ap"], batch["drr_lp"],
                     batch["P_ap"],   batch["P_lp"],
                     batch["center"], batch["scale"])

        loss, bd = total_loss(out, batch, epoch)

        # NaN guard
        if torch.isnan(loss) or torch.isinf(loss):
            nan_count += 1
            if nan_count <= 10:
                nan_keys = [k for k, v in bd.items() if math.isnan(v) or math.isinf(v)]
                print(f"  ⚠ Loss NaN/Inf in batch {bi}. Bad: {nan_keys}")
            if nan_count > 50:
                print(f"  ⚠ Too many NaN batches ({nan_count}), stopping epoch")
                break
            continue

        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

        if torch.isnan(grad_norm) or torch.isinf(grad_norm):
            nan_count += 1
            if nan_count <= 10:
                print(f"  ⚠ grad_norm={grad_norm:.1f} in batch {bi}. Skipping.")
            optimizer.zero_grad(set_to_none=True)
            continue

        optimizer.step()

        for k, v in bd.items(): acc[k] = acc.get(k, 0) + float(v)
        nb += 1

        if bi % 50 == 0 or bi == len(train_loader):
            g = lambda k: acc.get(k, 0) / max(1, nb)
            pbar.set_postfix_str(
                f"ch={g('chamfer'):.3f} gap={g('gap'):.4f} ax={g('axial'):.4f} "
                f"zg={g('zgap'):.4f} nan={nan_count}")

    scheduler.step()
    elapsed = time.time() - t0

    # ── Validation ──
    model.eval(); metrics = []; nd = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc='  Val', leave=False, ncols=100):
            if nd >= MAX_EVAL: break
            batch = to_dev(batch)
            out = model(batch["drr_ap"], batch["drr_lp"],
                        batch["P_ap"],   batch["P_lp"],
                        batch["center"], batch["scale"])
            pw = out["pred_world"].cpu().numpy()
            gw = batch["gt_ppc_world"].cpu().numpy()
            for b in range(pw.shape[0]):
                if nd >= MAX_EVAL: break
                metrics.append(chamfer_np(pw[b], gw[b])); nd += 1

    if metrics:
        mc = np.mean([m["chamfer_mm"]   for m in metrics])
        f2 = np.mean([m["fscore_2mm"]   for m in metrics])
        hd = np.mean([m["hausdorff_95"] for m in metrics])
        g  = lambda k: acc.get(k, 0) / max(1, nb)

        phase = "P1"
        if epoch >= PHASE3_EPOCH: phase = "P3"
        elif epoch >= PHASE2_EPOCH: phase = "P2"

        print(f"\n[Ep {epoch+1} {phase}] {elapsed:.0f}s  Val: Chamfer={mc:.3f}mm  F@2mm={f2:.3f}  HD95={hd:.3f}mm"
              f"\n  Train: ch={g('chamfer'):.3f} gap={g('gap'):.4f} ax={g('axial'):.4f}"
              f" zg={g('zgap'):.4f} nan={nan_count}")
        if epoch >= PHASE2_EPOCH:
            print(f"  P2: sw={g('sw'):.4f} proj={g('proj'):.4f} smooth={g('smooth'):.4f}"
                  f" spacing={g('spacing'):.6f} bnd={g('boundary'):.4f}")
        if epoch >= PHASE3_EPOCH:
            print(f"  P3: ivg={g('ivg'):.4f} vsep={g('vsep'):.4f} gw={g('gwidth'):.4f}")

        # Collapse detection
        if g("gap") >= 0.999:
            collapse_count += 1
            print(f"  ⚠ Collapse detected (gap≈1.0) — count {collapse_count}/5")
            if collapse_count >= 5 and BEST_CKPT.exists():
                print("  ⚠⚠ RESTORING best checkpoint, LR × 0.5")
                _, _, _ = load_ckpt(BEST_CKPT)
                for pg in optimizer.param_groups: pg["lr"] *= 0.5
                collapse_count = 0
                continue
        else:
            collapse_count = 0

        rec = {
            "epoch": epoch+1, "chamfer_mm": mc, "f2": f2, "hd95": hd,
            "train_gap": g("gap"), "train_axial": g("axial"),
            "train_zgap": g("zgap"), "train_chamfer": g("chamfer"),
            "nan_skips": nan_count, "phase": phase,
        }
        if epoch >= PHASE2_EPOCH:
            rec.update({"train_sw": g("sw"), "train_smooth": g("smooth"),
                        "train_spacing": g("spacing"), "train_boundary": g("boundary")})
        if epoch >= PHASE3_EPOCH:
            rec.update({"train_ivg": g("ivg"), "train_vsep": g("vsep"),
                        "train_gwidth": g("gwidth")})

        history.append(rec); append_log(TRAINING_LOG, rec)
        save_ckpt(CKPT_PATH, epoch, best_chamfer, history)

        if mc < best_chamfer:
            best_chamfer = mc; no_improve = 0
            save_ckpt(BEST_CKPT, epoch, best_chamfer, history)
            print(f"  ✓ New best: {best_chamfer:.3f} mm")
        else:
            no_improve += 1
            if no_improve >= EARLY_STOP_PATIENCE:
                print(f"  Early stop: {no_improve} epochs without improvement"); break

print(f"Done. Best val Chamfer: {best_chamfer:.3f} mm")

Params : 21.8 M
Training from SCRATCH — ImageNet encoder only.
Train  : 829 samples → 415 batches/ep
Val    : 103 samples → 52 batches/ep
Found checkpoint: latest_checkpoint.pth
  Resumed from ep 227 | best=1.788 mm
v12.2 training from ep 228/300
AMP=OFF | GRAD_CLIP=1.0 | fp32 everywhere
Phase 1: core only | Phase 2: ep 5+ | Phase 3: ep 15+


Ep 228/300: 100%|██████████████████████| 415/415 [01:22<00:00,  5.03it/s, ch=4.072 gap=0.8843 ax=0.0339 zg=0.0009 nan=0]



[Ep 228 P3] 83s  Val: Chamfer=1.795mm  F@2mm=0.702  HD95=3.579mm
  Train: ch=4.072 gap=0.8843 ax=0.0339 zg=0.0009 nan=0
  P2: sw=11.5779 proj=0.0000 smooth=0.0000 spacing=0.000001 bnd=0.1874
  P3: ivg=0.0002 vsep=0.0000 gw=0.0438
  Saved → latest_checkpoint.pth  (ep 228)


Ep 229/300: 100%|██████████████████████| 415/415 [01:00<00:00,  6.82it/s, ch=4.045 gap=0.8833 ax=0.0333 zg=0.0009 nan=0]



[Ep 229 P3] 61s  Val: Chamfer=1.807mm  F@2mm=0.697  HD95=3.601mm
  Train: ch=4.045 gap=0.8833 ax=0.0333 zg=0.0009 nan=0
  P2: sw=28.0223 proj=0.0000 smooth=0.0000 spacing=0.000001 bnd=0.1860
  P3: ivg=0.0002 vsep=0.0000 gw=0.0433
  Saved → latest_checkpoint.pth  (ep 229)


...


[Ep 285 P3] 86s  Val: Chamfer=1.800mm  F@2mm=0.701  HD95=3.605mm
  Train: ch=3.339 gap=0.8287 ax=0.0269 zg=0.0009 nan=0
  P2: sw=5.2966 proj=0.0000 smooth=0.0000 spacing=0.000001 bnd=0.1449
  P3: ivg=0.0002 vsep=0.0000 gw=0.0332
  Saved → latest_checkpoint.pth  (ep 285)


Ep 286/300: 100%|██████████████████████| 415/415 [01:00<00:00,  6.86it/s, ch=3.322 gap=0.8273 ax=0.0269 zg=0.0009 nan=0]



[Ep 286 P3] 60s  Val: Chamfer=1.798mm  F@2mm=0.702  HD95=3.598mm
  Train: ch=3.322 gap=0.8273 ax=0.0269 zg=0.0009 nan=0
  P2: sw=30.6888 proj=0.0000 smooth=0.0000 spacing=0.000001 bnd=0.1441
  P3: ivg=0.0002 vsep=0.0000 gw=0.0331
  Saved → latest_checkpoint.pth  (ep 286)


Ep 287/300: 100%|██████████████████████| 415/415 [01:19<00:00,  5.24it/s, ch=3.314 gap=0.8263 ax=0.0268 zg=0.0009 nan=0]



[Ep 287 P3] 79s  Val: Chamfer=1.798mm  F@2mm=0.701  HD95=3.589mm
  Train: ch=3.314 gap=0.8263 ax=0.0268 zg=0.0009 nan=0
  P2: sw=5.5909 proj=0.0000 smooth=0.0000 spacing=0.000001 bnd=0.1436
  P3: ivg=0.0002 vsep=0.0000 gw=0.0330
  Saved → latest_checkpoint.pth  (ep 287)
  Early stop: 60 epochs without improvement
Done. Best val Chamfer: 1.788 mm


In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# TEST EVALUATION
# ══════════════════════════════════════════════════════════════════════════════

print("Test evaluation...")
if BEST_CKPT.exists():
    st = torch.load(BEST_CKPT, map_location=device, weights_only=False)
    model.load_state_dict(st["model"])
    print(f"  Loaded best checkpoint from ep {st['epoch']+1} "
          f"(val Chamfer {st['best_chamfer']:.3f} mm)")
else:
    print("  WARNING: No best checkpoint — using current model state.")

model.eval()
test_loader = DataLoader(LumbarDataset(test_ids, aug=False),
                         batch_size=2, shuffle=False, num_workers=NUM_WORKERS,
                         pin_memory=True, collate_fn=collate_fn)
all_m = []

with torch.no_grad():
    for batch in test_loader:
        batch = to_dev(batch)
        out = model(batch["drr_ap"], batch["drr_lp"],
                    batch["P_ap"],   batch["P_lp"],
                    batch["center"], batch["scale"])
        pw   = out["pred_world"].cpu().numpy()
        gt   = batch["gt_ppc_world"].cpu().numpy()
        pids = batch["patient_id"]
        for b in range(pw.shape[0]):
            m = chamfer_np(pw[b], gt[b]); m["patient_id"] = pids[b]
            all_m.append(m)
            save_vtk_points(pw[b], RESULTS_DIR / f"{pids[b]}_pred.vtk")

print(f"{'='*65}")
print(f"TEST RESULTS  ({len(all_m)} patients)")
print(f"{'='*65}")
for key, lbl in [("chamfer_mm","Chamfer mm  "), ("fscore_1mm","F-score @1mm"),
                 ("fscore_2mm","F-score @2mm"), ("fscore_5mm","F-score @5mm"),
                 ("hausdorff_95","HD95 mm     ")]:
    vals = [m[key] for m in all_m]
    print(f"  {lbl}  mean={np.mean(vals):.3f}  std={np.std(vals):.3f}  "
          f"median={np.median(vals):.3f}  p95={np.percentile(vals,95):.3f}")

csv_path = RESULTS_DIR / "test_results_v12_2.csv"
with open(csv_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["patient_id","chamfer_mm",
                                      "fscore_1mm","fscore_2mm","fscore_5mm","hausdorff_95"])
    w.writeheader(); w.writerows(all_m)
print(f"CSV → {csv_path}")

Test evaluation...
  Loaded best checkpoint from ep 119 (val Chamfer 1.788 mm)
TEST RESULTS  (105 patients)
  Chamfer mm    mean=1.999  std=1.042  median=1.793  p95=2.832
  F-score @1mm  mean=0.144  std=0.067  median=0.138  p95=0.265
  F-score @2mm  mean=0.659  std=0.161  median=0.680  p95=0.878
  F-score @5mm  mean=0.972  std=0.077  median=0.994  p95=1.000
  HD95 mm       mean=4.474  std=5.001  median=3.425  p95=6.937
CSV → /apps/app/pandu/ppc_network_v12_2/results/test_results_v12_2.csv
